# Vectorless RAG Pipeline (PageIndex) — FinanceBench (full batch run)

**Paper:** Lumer et al. (2025), arXiv 2511.18177
**Library:** `pageindex` (PyPI, pinned `0.2.12`) — installed via pip, not the
vendored `external/PageIndex` git clone the previous version of this notebook
used. That folder is gitignored ("cloned, not vendored"), so it never existed
on a fresh Colab clone; the pip package covers the same functions
(`page_index_main`, `ConfigLoader`, `get_page_tokens`, `create_node_mapping`,
`llm_completion`) at the top level. See `vectorless_rag_walkthrough.ipynb` for
the debugging trail that found this, plus the asyncio/rate-limit issues below.

## What this pipeline is

No embeddings, no chunking. Each document is parsed once into a hierarchical
tree index (a table-of-contents structure with per-section titles and short
LLM-written summaries), and an LLM navigates that tree to pick the section(s)
most likely to hold the answer — no vector similarity search involved. Only
the raw text of the *selected* sections is passed to the generator. Failure
mode per CLAUDE.md: incorrect navigation to the wrong section.

## Scope decisions carried over from the walkthrough

- **Model: `gemini-3.1-flash-lite`, not `gemini-3.5-flash`.** Free tier caps
  `gemini-3.5-flash` at 5 requests/minute, and tree-building fires many
  concurrent calls — hit that wall directly in the walkthrough. flash-lite has
  a much bigger free-tier allowance. This also matches `vector_rag_pipeline.ipynb`,
  which switched for the same reason — keeping the same generation model across
  pipelines matters for a fair accuracy-latency-cost comparison.
- **`nest_asyncio.apply()` is required.** `page_index_main` calls its own
  `asyncio.run()` internally; a notebook kernel already has one running, so
  without this patch tree-building raises
  `RuntimeError: asyncio.run() cannot be called from a running event loop`.
- **Parser: PyPDF2 (PageIndex's default)**, not pymupdf4llm. Lower text
  quality on tables, but no extra dependency, and it's the parser
  `page_index_main` already uses internally to decide section page ranges —
  matching it here means no risk of page boundaries desyncing.
- **Resumable.** Every document/question already saved to disk is skipped on
  rerun. A real 84-document, 150-question run realistically won't finish in
  one sitting, and free-tier rate limits mean some calls will fail outright.
- **`gemini-3.1-flash-lite` billed at $0.25/1M input, $1.50/1M output**
  (confirmed against ai.google.dev/gemini-api/docs/pricing) — real money if
  billing is attached to the API key, since Google may bill directly instead
  of falling back to the free tier once a payment method is on the account.
  Use a billing-free key to actually get the free tier.

## Pipeline stages

```
Stage 0  Setup            — Drive mount, repo clone/pull, API keys, output paths
Stage 1  Load data        — 150 questions, 84 unique documents
Stage 2  Build trees      — PDF -> hierarchical section tree, one per document (resumable)
Stage 3  Navigate         — LLM picks node_id(s) likely to hold the answer (resumable)
Stage 4  Generate         — raw text of selected node(s) + question -> answer (resumable)
Stage 5  Preview          — sample of generated answers next to gold answers
Stage 6  Score            — deterministic match + LLM judge fallback (resumable)
Stage 7  Retrieval        — Recall@k / MRR@k against gold evidence pages (resumable, no cost)
Stage 8  Latency          — dedicated sequential timing pass, median-of-N (resumable, costs extra)
Stage 9  Summarize        — answer quality + retrieval quality + token/cost totals
```

Stages 6-9 reuse `evaluation/answer_scorer.py`, `evaluation/retrieval_metrics.py`,
and `evaluation/cost_tracker.py` — the same shared modules `vector_rag_pipeline.ipynb`
uses, so both pipelines are scored, measured, and costed identically.

---
## Stage 0 — Setup

In [1]:
import os, sys, json, re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive')

REPO_ROOT = Path('/content/drive/MyDrive/financebench_project')
if not (REPO_ROOT / "data" / "financebench_open_source.jsonl").exists():
    print(f"Repo not found at {REPO_ROOT} — cloning ...")
    !git clone https://github.com/shaliqsv/financebench-rag-thesis.git "{REPO_ROOT}"
else:
    print(f"Repo already present at {REPO_ROOT} — syncing to latest main ...")
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" checkout main
    !git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit

sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
PDF_DIR  = REPO_ROOT / "pdfs"

RESULTS_DIR      = REPO_ROOT / "experiments" / "results"
TREE_DIR         = RESULTS_DIR / "vectorless_rag_index"          # one tree JSON per document
NAVIGATION_PATH  = RESULTS_DIR / "vectorless_rag_stage_navigation.jsonl"
GENERATION_PATH  = RESULTS_DIR / "vectorless_rag_stage_generation.jsonl"
SCORING_PATH     = RESULTS_DIR / "vectorless_rag_stage_scoring.jsonl"
RETRIEVAL_PATH   = RESULTS_DIR / "vectorless_rag_stage_retrieval.jsonl"
COSTS_PATH       = RESULTS_DIR / "vectorless_rag_costs.jsonl"
INDEXING_LATENCY_PATH = RESULTS_DIR / "vectorless_rag_indexing_latency.jsonl"  # per-document wall time, method-tagged (classic vs flash)
LATENCY_PATH     = RESULTS_DIR / "vectorless_rag_latency.jsonl"

# override=True: without it, re-running this cell after editing .env keeps the
# stale value already sitting in the kernel's os.environ from an earlier run.
load_dotenv(REPO_ROOT / ".env", override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY", "")
print("GOOGLE_API_KEY:", "ok" if GOOGLE_API_KEY else "MISSING - fill in .env")
print("GROQ_API_KEY  :", "ok" if GROQ_API_KEY else "MISSING - fill in .env (needed for Stage 6's LLM judge)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already present at /content/drive/MyDrive/financebench_project — syncing to latest main ...
D	experiments/results/vector_rag_results.jsonl
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/shaliqsv/financebench-rag-thesis
 * branch            main       -> FETCH_HEAD
Already up to date.
GOOGLE_API_KEY: ok
GROQ_API_KEY  : ok


In [2]:
# pinned, not "latest pageindex" -- so this notebook keeps behaving the same
# way weeks from now instead of picking up whatever VectifyAI ships next.
# pycryptodome: PyPDF2 needs it to read AES-encrypted PDFs (hit this on
# ADOBE_2022_10K -- some SEC filings are encrypted, not just password-locked).
%pip install -q python-dotenv pandas litellm PyPDF2 pyyaml pageindex==0.2.12 nest_asyncio groq pycryptodome

In [3]:
import litellm
import nest_asyncio
from pageindex import ConfigLoader, get_page_tokens, create_node_mapping, llm_completion, page_index_main

litellm.drop_params = True

# page_index_main runs its own asyncio.run() internally, but this notebook's
# kernel is already inside a running event loop -- nest_asyncio patches the
# loop so a nested asyncio.run() is allowed instead of raising RuntimeError.
nest_asyncio.apply()

# litellm routes by prefix ("gemini/...") and its Gemini provider reads
# GOOGLE_API_KEY natively -- no extra wiring needed.
MODEL = "gemini/gemini-3.1-flash-lite"
opt = ConfigLoader().load({"model": MODEL})
opt

namespace(toc_check_page_num=20,
          max_page_num_each_node=10,
          max_token_num_each_node=20000,
          if_add_node_id='yes',
          if_add_node_summary='yes',
          if_add_doc_description='no',
          if_add_node_text='no',
          model='gemini/gemini-3.1-flash-lite',
          index_model='gemini/gemini-3.1-flash-lite',
          summary_model='gemini/gemini-3.1-flash-lite',
          chat_model='gemini/gemini-3.1-flash-lite',
          retrieve_model='gemini/gemini-3.1-flash-lite')

---
## Cost tracking

Logs every LLM call's token counts (and cost, since `gemini-3.1-flash-lite`'s
price is filled in) to `COSTS_PATH`. `llm_completion`/`llm_acompletion` are
patched in two places: once for this notebook's own calls
(`navigate_tree`/`generate_answer`), and once inside `pageindex.page_index_classic`
— the module `page_index_main` calls internally for TOC detection,
verification, and summaries during Stage 2. Patching only the first wouldn't
see any of Stage 2's indexing cost, since that module imported its own copy
of these functions at import time (`from .utils import *`), so it isn't
looking at the same name our own `from pageindex import llm_completion`
bound.

Token counts are estimated locally via `litellm.token_counter` on the
prompt/response text, since `llm_completion` only returns the generated
text, not a usage object — not billing-exact, but the same approach the
walkthrough already validated for page-token counting. Stage 6's judge cost
is exact instead (Groq's API returns real usage).

In [4]:
import asyncio

from evaluation.cost_tracker import CostTracker, PRICING_PER_MILLION_TOKENS

cost_tracker = CostTracker(COSTS_PATH)

_current_doc_name = None
_current_financebench_id = None
_current_stage = None

_raw_llm_completion = llm_completion


def _log_llm_usage(model, prompt, response, latency_sec):
    # response is a plain string normally, but PageIndex sometimes calls
    # llm_completion(..., return_finish_reason=True), which returns a
    # (content, finish_reason) tuple instead -- litellm.token_counter only
    # accepts a string or list of strings, so pull the text out first.
    content = response[0] if isinstance(response, tuple) else response
    input_tokens = litellm.token_counter(model=model, text=prompt)
    output_tokens = litellm.token_counter(model=model, text=content)
    cost_tracker.log(
        pipeline="vectorless_rag",
        stage=_current_stage or "unknown",
        model=model.removeprefix("gemini/"),  # matches PRICING_PER_MILLION_TOKENS' key style
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        doc_name=_current_doc_name,
        financebench_id=_current_financebench_id,
        latency_sec=latency_sec,
    )


# Redefining llm_completion at the top level makes navigate_tree/generate_answer
# use this wrapped version automatically -- they look up the bare name
# "llm_completion" fresh on every call, not the object that existed when they
# were defined. Timed here too, so every call -- indexing or per-query --
# gets its own latency logged next to its own cost, not just a per-document
# or per-question total.
def llm_completion(model, prompt, *args, **kwargs):
    t0 = time.time()
    response = _raw_llm_completion(model, prompt, *args, **kwargs)
    _log_llm_usage(model, prompt, response, time.time() - t0)
    return response


# PageIndex's own tree-building calls use its own copy of llm_completion/
# llm_acompletion (imported at module load time) -- patching the module's
# names directly is the only way to see that cost.
import pageindex.page_index_classic as _pageindex_internal

_raw_llm_acompletion = _pageindex_internal.llm_acompletion
_pageindex_internal.llm_completion = llm_completion

# PageIndex's node-summary step (generate_summaries_for_structure, what
# page_index_main actually calls) fires every section's summary call at once
# via asyncio.gather with no concurrency cap of its own -- a large filing with
# many sections means dozens of simultaneous but hwarequests in one burst, which is
# exactly the kind of load that trips a transient 503 from Gemini. Throttling
# here, at the one point every async call already passes through, caps that
# burst without needing to touch PageIndex's own code. Lower = fewer 503s but
# slower; higher = faster but more overload risk.
LLM_ACOMPLETION_CONCURRENCY = 10
_llm_semaphore = asyncio.Semaphore(LLM_ACOMPLETION_CONCURRENCY)


# PageIndex's own retry loop (inside _raw_llm_acompletion) only retries on a
# raised exception -- it treats a "successful" call that comes back with
# empty content as done, not failed. Gemini does occasionally return
# finish_reason="stop" with an empty message, especially under concurrent
# load, and that's what took out every node in a document at once with no
# error or retry logged anywhere: nothing ever raised, so nothing ever
# retried. This wrapper adds the retry PageIndex's own code is missing,
# specifically for that empty-but-"successful" case.
_EMPTY_RESPONSE_RETRIES = 3


async def _tracked_llm_acompletion(model, prompt):
    for attempt in range(_EMPTY_RESPONSE_RETRIES):
        t0 = time.time()
        async with _llm_semaphore:
            response = await _raw_llm_acompletion(model, prompt)
        _log_llm_usage(model, prompt, response, time.time() - t0)
        content = response[0] if isinstance(response, tuple) else response
        if content:
            return response
        if attempt < _EMPTY_RESPONSE_RETRIES - 1:
            print(f"    (empty response from {model}, retrying {attempt + 1}/{_EMPTY_RESPONSE_RETRIES}) ...")
            await asyncio.sleep(1)
    return response


_pageindex_internal.llm_acompletion = _tracked_llm_acompletion


---
## Stage 1 — Load FinanceBench data

All 150 questions across 84 unique source documents (some documents have multiple questions).

In [5]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta      = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df           = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

doc_names = sorted(df.doc_name.unique())

print(f"Total questions : {len(df)}")
print(f"Unique documents: {len(doc_names)}")
df[["financebench_id", "doc_name", "question"]].head(3)

Total questions : 150
Unique documents: 84


,financebench_id,doc_name,question
0,financebench_id_03029,3M_2018_10K,What is the FY2018 capital expenditure amount ...
1,financebench_id_04672,3M_2018_10K,Assume that you are a public equities analyst....
2,financebench_id_00499,3M_2022_10K,Is 3M a capital-intensive business based on FY...


---
## Stage 2 — Build the PageIndex tree for every document

Parses each PDF into a hierarchical section tree once (titles + node_ids +
LLM-written summaries) and separately caches each page's raw text, so Stage 4
can pull a selected node's text later without re-parsing the PDF. Saved to
`TREE_DIR/{doc_name}_tree.json` — **resumable**, a document already indexed
is skipped.

In [6]:
import time

# Read back the results file saved so far and pull out which question IDs
# are already done, so the batch loops below know what to skip.
def _load_jsonl(path) -> list[dict]:
    if not path.exists():
        return []
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


def _load_completed_ids(path) -> set:
    return {r["financebench_id"] for r in _load_jsonl(path)}


# Find, check for, and read back one document's saved tree file -- needed
# because trees now get saved to disk instead of just staying in memory.
def tree_path(tree_dir, doc_name):
    return tree_dir / f"{doc_name}_tree.json"


def is_tree_built(tree_dir, doc_name) -> bool:
    return tree_path(tree_dir, doc_name).exists()


def load_tree(tree_dir, doc_name) -> dict:
    return json.loads(tree_path(tree_dir, doc_name).read_text())


# Total wall-clock time to build one document's tree, method-tagged so
# classic and flash indexing runs stay comparable -- separate from the
# per-call latency the cost tracker logs below, since the concurrency cap on
# LLM calls (see LLM_ACOMPLETION_CONCURRENCY) means a document's total time
# isn't just the sum of its calls' individual latencies.
def _log_indexing_latency(doc_name, method, elapsed_sec, n_pages, n_top_level_sections):
    INDEXING_LATENCY_PATH.parent.mkdir(parents=True, exist_ok=True)
    with INDEXING_LATENCY_PATH.open("a") as f:
        f.write(json.dumps({
            "doc_name": doc_name,
            "method": method,
            "elapsed_sec": elapsed_sec,
            "n_pages": n_pages,
            "n_top_level_sections": n_top_level_sections,
        }) + "\n")


In [7]:
# Walkthrough's Stage 1, exactly: build the tree, cache the page text.
# Wrapped in a function because this now runs once per document (84 times)
# instead of once for a single example. Timed end to end (TOC detection +
# verification + summaries + the PDF page-text cache) and logged to
# INDEXING_LATENCY_PATH, method="classic" -- the project brief's
# preprocessing-latency-reported-separately metric.
def build_tree(pdf_path, doc_name, opt) -> dict:
    t0 = time.time()
    result = page_index_main(str(pdf_path), opt)
    page_list = get_page_tokens(str(pdf_path), model=opt.model)
    elapsed = time.time() - t0
    record = {
        "doc_name": doc_name,
        "structure": result["structure"],
        "page_texts": [p[0] for p in page_list],
        "n_pages": len(page_list),
    }
    _log_indexing_latency(doc_name, "classic", elapsed, record["n_pages"], len(record["structure"]))
    return record


In [8]:
# Which documents Stage 2 didn't leave a tree file for -- these were either
# never attempted yet or hit a FAILED in the loop above and got skipped.
missing_tree_docs = [d for d in doc_names if not is_tree_built(TREE_DIR, d)]

print(f"{len(doc_names) - len(missing_tree_docs)}/{len(doc_names)} documents indexed")
if missing_tree_docs:
    print(f"Missing ({len(missing_tree_docs)}):")
    for d in missing_tree_docs:
        print(f"  {d}")
else:
    print("All documents indexed.")

70/84 documents indexed
Missing (14):
  ACTIVISIONBLIZZARD_2019_10K
  ADOBE_2022_10K
  AES_2022_10K
  AMERICANWATERWORKS_2022_10K
  BOEING_2022_10K
  CVSHEALTH_2018_10K
  JPMORGAN_2021Q1_10Q
  JPMORGAN_2022_10K
  PEPSICO_2021_10K
  PEPSICO_2022_10K
  PEPSICO_2023_8K_dated-2023-05-30
  PFIZER_2021_10K
  WALMART_2018_10K
  WALMART_2019_10K


---
## Stage 2b — Rebuild every document with PageIndex Flash

The classic run above (`page_index_main`) fails outright on some documents
and never even reached others: `page_index_main`'s TOC step asks the LLM to
return a strict, exactly-matching JSON table of contents, and
`gemini-3.1-flash-lite` isn't reliable enough at that strict a format to pull
it off consistently (`LLM returned a different number of TOC entries`,
`LLM returned reordered or modified TOC entries`, `FAILED: 'physical_index'`),
and a further 21 documents were never even attempted before the run got cut
off (likely a Colab disconnect).

**PageIndex Flash** (`pageindex.flash.page_index_flash`, shipped in the same
`pageindex==0.2.12` package and now the library's own documented default —
its own benchmark is run on Flash, not the classic path) sidesteps the bug
entirely: the tree structure comes from the PDF's own layout info (fonts,
headings, embedded bookmarks), not an LLM guessing a TOC in strict JSON, so
there is no strict-format-matching step left to fail. An LLM is only asked
for per-node *summaries* afterward — a far easier, less failure-prone task.

**Runs against all 84 documents, saved to a separate folder —
`TREE_DIR_FLASH`, not `TREE_DIR`.** This intentionally re-indexes the 60+
documents already done the classic way too, so both trees are built the same
way end to end and comparable — nothing is read from or written to `TREE_DIR`,
so the classic run's output is completely untouched no matter what happens
here. That does mean paying for indexing all 84 documents again — this is a
second full Stage 2 pass, not a top-up of the missing ones.

Downstream stages (3 onward) still read from `TREE_DIR` — pointing them at
`TREE_DIR_FLASH` instead is a separate step once these trees are checked over.

In [9]:
import sys

# Full run (build_all_trees_flash, 84 docs) hit 'maximum recursion depth
# exceeded' partway through -- not on any specific document (an isolated
# single-doc call to build_tree_flash on one of the 'failing' PDFs worked
# fine), so it looks like something cumulative across many page_index_flash()
# calls in one process rather than a bad PDF -- nest_asyncio.apply() (cell 4)
# lets page_index_flash's internal asyncio.run() nest inside this kernel's
# already-running loop once per document, a known way for nested-asyncio
# setups to build up call-stack depth across iterations instead of
# unwinding it. Raising the ceiling well above the default 1000 is a
# stopgap so the loop survives, not a fix for the underlying nesting.
sys.setrecursionlimit(5000)

from pageindex.flash import page_index_flash
from pageindex.utils import write_node_id

# Flash's own structure extraction never calls an LLM, but its node-summary
# step (summarize_tree, in pageindex/utils.py) and its tree-optimize step
# (pageindex/tree_optimize.py) both do -- and neither module is the
# page_index_classic module already patched above, so their calls would be
# invisible to cost_tracker without patching them too. Each module imported
# its own copy of llm_acompletion at import time (same reason the classic
# patch above targets page_index_classic specifically, not utils directly),
# so both need their own module-level name repointed -- reusing the exact
# same _tracked_llm_acompletion wrapper (same semaphore, same cost log)
# defined above rather than building a second one.
import pageindex.utils as _pageindex_utils
import pageindex.tree_optimize as _pageindex_tree_optimize

_pageindex_utils.llm_acompletion = _tracked_llm_acompletion
_pageindex_tree_optimize.llm_acompletion = _tracked_llm_acompletion

# Flash's PDF-layout parser (extract_toc, called by page_index_flash) hands
# off to a pool of worker *processes* for any document with 64+ pages --
# nearly every 10-K/10-Q here clears that. Spawning worker processes from
# inside a notebook kernel (Colab included) is a known way for this to hang
# silently and indefinitely: no error, no output, the cell just never
# returns. PageIndex's own fallback only catches a pool that raises an
# error -- not one that just hangs, which is what happened here.
#
# extract_toc's own `workers=1` argument forces its single-process path
# instead, but page_index_flash() never exposes that argument, so the only
# way to set it is patching the function main.py actually calls. main.py did
# `from .parser_pdfium_parallel import parse_charlevel_meta_parallel` --an
# explicit import, same reason as the llm_acompletion patches above -- so
# `pageindex.flash.main`, not `parser_pdfium_parallel`, is the module that
# needs its copy of the name repointed.
import pageindex.flash.main as _pageindex_flash_main

_raw_parse_charlevel_meta_parallel = _pageindex_flash_main.parse_charlevel_meta_parallel


def _parse_charlevel_meta_sequential_only(doc_handle, workers=None):
    return _raw_parse_charlevel_meta_parallel(doc_handle, workers=1)


_pageindex_flash_main.parse_charlevel_meta_parallel = _parse_charlevel_meta_sequential_only

TREE_DIR_FLASH = RESULTS_DIR / "vectorless_rag_index_flash"  # separate from TREE_DIR -- keeps the classic run's trees untouched

In [10]:
# summary=True asks the model for per-node summaries (matching classic's
# if_add_node_summary='yes'); optimize="merge" runs Flash's deterministic
# node-merging pass but skips its optional LLM "expand" pass -- expand tunes
# retrieval quality further but classic's page_index_main never did anything
# equivalent either, so leaving it off keeps this a fairer comparison and
# one fewer LLM call per document.
#
# summary_concurrency=8 caps simultaneous summary calls per document. The
# library default (64) fires every node at Gemini at once -- that burst is
# what triggered the 503 "high demand" pileup that failed 3M_2018_10K's
# whole tree (10 retries x 64 nodes, 1s apart, no backoff). 8 spreads the
# same work out instead of bursting it.
#
# page_index_flash() doesn't assign node_id itself (only its embedded-bookmark
# path does) -- write_node_id() is what page_index_main() calls internally
# for the same purpose, so calling it here keeps node_ids compatible with
# create_node_mapping/navigate_tree/fetch_node_text exactly as before.
#
# Timed end to end (structure extraction + summaries) and logged to
# INDEXING_LATENCY_PATH, method="flash" -- same metric as classic's
# build_tree, so the two are directly comparable.
def build_tree_flash(pdf_path, doc_name, model) -> dict:
    t0 = time.time()
    result = page_index_flash(str(pdf_path), summary=True, summary_model=model, optimize="merge", summary_concurrency=8)
    structure = result["structure"]
    write_node_id(structure)
    page_list = get_page_tokens(str(pdf_path), model=model)
    elapsed = time.time() - t0
    record = {
        "doc_name": doc_name,
        "structure": structure,
        "page_texts": [p[0] for p in page_list],
        "n_pages": len(page_list),
    }
    _log_indexing_latency(doc_name, "flash", elapsed, record["n_pages"], len(record["structure"]))
    return record


In [14]:
# Smoke test: run build_tree_flash on one already-known-good document
# before trusting the full 84-doc loop again. Picks one of the docs
# that failed with 'maximum recursion depth exceeded' last run --
# if this single call succeeds on its own, that confirms the failure
import pageindex.utils as pu
_orig = pu.llm_acompletion

async def _patched(model, prompt):
    try:
        r = await _orig(model, prompt)
        if not r:
            print("EMPTY REPLY from model")
        return r
    except Exception as e:
        print("REAL ERROR:", type(e).__name__, getattr(e, "status_code", None), str(e)[:400])
        raise
    

In [15]:
# Smoke test: run build_tree_flash on one already-known-good document
# before trusting the full 84-doc loop again. Picks one of the docs
# that failed with 'maximum recursion depth exceeded' last run --
# if this single call succeeds on its own, that confirms the failure
# was cumulative (see the sys.setrecursionlimit comment above), not
# something wrong with this specific PDF or with build_tree_flash itself.
_test_doc = "WALMART_2018_10K"
_test_record = build_tree_flash(PDF_DIR / f"{_test_doc}.pdf", _test_doc, MODEL)
print(f"OK -- {_test_record['n_pages']} pages, "
      f"{len(_test_record['structure'])} top-level sections")

RuntimeError: Summary generation failed for all nodes (every summary call failed or returned empty; check the model and its context limits)

In [ ]:
# Diagnostic: instrument llm_acompletion so a failed or empty summary call
# prints its real error instead of being silently swallowed into the
# generic "Summary generation failed for all nodes" message, then run
# page_index_flash directly on one document to see what actually happens.
import pageindex.utils as pu
_orig_llm_acompletion = pu.llm_acompletion

async def _patched(model, prompt):
    try:
        r = await _orig_llm_acompletion(model, prompt)
        if not r:
            print("EMPTY REPLY from model")
        return r
    except Exception as e:
        print("REAL ERROR:", type(e).__name__, getattr(e, "status_code", None), str(e)[:400])
        raise

pu.llm_acompletion = _patched

_diag_doc = "WALMART_2018_10K"
pdf_path = str(PDF_DIR / f"{_diag_doc}.pdf")
model = MODEL

result = page_index_flash(
    pdf_path,
    summary=True,
    summary_model=model,
    optimize=True,
    optimize_expand=False,
)

print(result)


In [12]:
import litellm
try:
    r = litellm.completion(model="gemini/gemini-3.1-flash-lite", messages=[{"role":"user","content":"hi"}], max_retries=0)
    print("OK:", r.choices[0].message.content)
except Exception as e:
    print(type(e).__name__, getattr(e, "status_code", None), str(e)[:500])

OK: Hello! How can I help you today?


In [10]:
# Same loop shape as build_all_trees above, pointed at TREE_DIR_FLASH and
# run over every document (doc_names), not just the missing ones -- an
# independent, complete Stage 2 pass. Resumable, same as the classic loop:
# safe to re-run after a disconnect without re-paying for documents already
# done here.
def build_all_trees_flash(doc_names, pdf_dir, tree_dir, model):
    global _current_doc_name, _current_stage
    tree_dir.mkdir(parents=True, exist_ok=True)
    for i, doc_name in enumerate(doc_names, start=1):
        if is_tree_built(tree_dir, doc_name):
            print(f"[{i}/{len(doc_names)}] {doc_name}: already indexed (flash), skipping")
            continue
        print(f"[{i}/{len(doc_names)}] {doc_name}: building tree (flash) ...")
        _current_doc_name, _current_stage = doc_name, "indexing_flash"
        try:
            record = build_tree_flash(pdf_dir / f"{doc_name}.pdf", doc_name, MODEL)
            tree_path(tree_dir, doc_name).write_text(json.dumps(record))
            print(f"    done -- {record['n_pages']} pages, {len(record['structure'])} top-level sections")
        except Exception as e:
            print(f"    FAILED: {e}")


build_all_trees_flash(doc_names, PDF_DIR, TREE_DIR_FLASH, MODEL)

[1/84] 3M_2018_10K: building tree (flash) ...


KeyboardInterrupt: 

In [ ]:
# Same shape as the missing_tree_docs check earlier, but against
# TREE_DIR_FLASH and all 84 documents -- how the Flash rebuild did overall,
# independent of how the classic run (still sitting untouched in TREE_DIR) did.
missing_tree_docs_flash = [d for d in doc_names if not is_tree_built(TREE_DIR_FLASH, d)]

print(f"{len(doc_names) - len(missing_tree_docs_flash)}/{len(doc_names)} documents indexed via Flash")
if missing_tree_docs_flash:
    print(f"Missing ({len(missing_tree_docs_flash)}):")
    for d in missing_tree_docs_flash:
        print(f"  {d}")
else:
    print("All documents indexed via Flash.")

In [ ]:
# Compares classic vs Flash indexing on the two axes that matter for the
# thesis' cost-latency-accuracy frontier: preprocessing cost (from
# COSTS_PATH, stage="indexing" for classic vs "indexing_flash" for Flash)
# and preprocessing latency (from INDEXING_LATENCY_PATH, method-tagged).
import statistics

def summarize_indexing(latency_path, cost_path):
    latency_records = _load_jsonl(latency_path)
    costs = _load_jsonl(cost_path)
    for method, stage_name in [("classic", "indexing"), ("flash", "indexing_flash")]:
        durations = [r["elapsed_sec"] for r in latency_records if r["method"] == method]
        stage_costs = [r["cost_usd"] for r in costs if r["stage"] == stage_name and r["cost_usd"] is not None]
        n_calls = sum(1 for r in costs if r["stage"] == stage_name)
        print(f"{method} ({stage_name}):")
        print(f"  documents timed   : {len(durations)}")
        if durations:
            print(f"  median doc time   : {statistics.median(durations):.1f}s")
        print(f"  LLM calls logged  : {n_calls}")
        if stage_costs:
            print(f"  total cost        : ${sum(stage_costs):.4f}")
        print()


summarize_indexing(INDEXING_LATENCY_PATH, COSTS_PATH)

---
## Stage 3 — Navigate the tree (select relevant sections)

One LLM call per question: the whole tree (titles, node_ids, summaries — no
section text) is shown at once, and the model returns a ranked JSON array of
node_ids likely to contain the answer. If parsing fails or the model returns
nothing usable, this falls back to *every* node in the tree rather than
silently generating from zero context in Stage 4.

**Resumable** — needs Stage 2 to have built the document's tree first.

In [17]:
NAVIGATION_PROMPT = """You are navigating a financial filing's table-of-contents-style section tree \
to find the section(s) most likely to contain the answer to a question. You cannot see the section \
text yet, only titles and summaries. Return ONLY a JSON array of node_id strings, ordered from most \
to least likely to contain the answer (e.g. ["0003", "0007"]). Include at most 5 node_ids.

Question: {question}

Document tree:
{tree_outline}"""


# Walkthrough's Stage 2, exactly: build the prompt, ask the model, pull the
# node_ids out of the reply. One addition -- if the model's answer can't be
# parsed or comes back empty, fall back to every node in the tree, so Stage 4
# never generates from zero information.
def navigate_tree(question, structure, model) -> list:
    node_map = create_node_mapping(structure)
    prompt = NAVIGATION_PROMPT.format(question=question, tree_outline=json.dumps(structure, indent=2))
    response = llm_completion(model, prompt)
    match = re.search(r"\[.*\]", response, re.DOTALL)
    node_ids = json.loads(match.group(0)) if match else []
    if not node_ids:
        node_ids = list(node_map.keys())  # fallback: whole document
    return node_ids

In [ ]:
# The loop over all 150 questions: skip ones already navigated, skip ones
# whose document isn't indexed yet, otherwise call navigate_tree and save
# the result.
def run_navigation_all(df, tree_dir, out_path, model):
    global _current_doc_name, _current_financebench_id, _current_stage
    out_path.parent.mkdir(parents=True, exist_ok=True)
    completed = _load_completed_ids(out_path)
    tree_cache = {}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            continue
        if not is_tree_built(tree_dir, row.doc_name):
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- {row.doc_name} not indexed yet")
            continue

        print(f"[{i}/{len(df)}] {fb_id}: navigating ...")
        _current_doc_name, _current_financebench_id, _current_stage = row.doc_name, fb_id, "navigation"
        try:
            if row.doc_name not in tree_cache:
                tree_cache[row.doc_name] = load_tree(tree_dir, row.doc_name)
            structure = tree_cache[row.doc_name]["structure"]
            node_ids = navigate_tree(row.question, structure, model)
            record = {"financebench_id": fb_id, "doc_name": row.doc_name, "node_ids": node_ids}
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
        except Exception as e:
            print(f"    FAILED: {e}")


run_navigation_all(df, TREE_DIR, NAVIGATION_PATH, MODEL)

[1/150] financebench_id_03029: navigating ...
[2/150] financebench_id_04672: navigating ...
[3/150] financebench_id_00499: navigating ...
[4/150] financebench_id_01226: navigating ...
[5/150] financebench_id_01865: navigating ...
[6/150] financebench_id_00807: navigating ...
[7/150] financebench_id_00941: navigating ...
[8/150] financebench_id_01858: navigating ...
[9/150] financebench_id_02987: SKIPPED -- ACTIVISIONBLIZZARD_2019_10K not indexed yet
[10/150] financebench_id_07966: SKIPPED -- ACTIVISIONBLIZZARD_2019_10K not indexed yet
[11/150] financebench_id_04735: navigating ...
[12/150] financebench_id_07507: navigating ...
[13/150] financebench_id_03856: navigating ...
[14/150] financebench_id_00438: SKIPPED -- ADOBE_2022_10K not indexed yet
[15/150] financebench_id_00591: SKIPPED -- ADOBE_2022_10K not indexed yet
[16/150] financebench_id_01319: SKIPPED -- AES_2022_10K not indexed yet
[17/150] financebench_id_00540: SKIPPED -- AES_2022_10K not indexed yet
[18/150] financebench_id_1

---
## Stage 4 — Generate the answer from the selected sections' raw text

Only the raw text of the node(s) Stage 3 selected is passed to the generator
— never the whole document. Extracted directly from Stage 2's cached
`page_texts` by the node's `start_index`/`end_index` (1-indexed, inclusive).

**Resumable** — needs Stage 3 to have produced a navigation record for the
question first.

In [ ]:
# Walkthrough's Stage 3, exactly: pull out only the raw text of the
# sections navigation picked.
def fetch_node_text(node_ids, node_map, page_texts) -> str:
    parts = []
    for node_id in node_ids:
        node = node_map.get(node_id)
        if node is None:
            continue
        start, end = node["start_index"], node["end_index"]
        text = "".join(page_texts[start - 1:end])
        parts.append(f"[Section: {node['title']}]\n{text}")
    return "\n\n".join(parts)

In [ ]:
GENERATION_PROMPT = """You are a financial analyst answering a question using only the sections \
below from a company's SEC filing. Answer concisely and precisely, matching the format the question \
expects (a number, a yes/no with brief reasoning, etc). If the sections don't contain enough \
information to answer, say so explicitly rather than guessing.

Question: {question}

Sections:
{sections}

Answer:"""


# Walkthrough's Stage 4, exactly: build the prompt, ask the model, return
# the answer.
def generate_answer(question, sections_text, model) -> str:
    prompt = GENERATION_PROMPT.format(question=question, sections=sections_text)
    return llm_completion(model, prompt)

In [ ]:
# The loop: for each question that has a navigation result but no answer
# yet, load its document's tree (cached so it's not re-read for every
# question on the same doc), fetch the section text, generate, save.
def run_generation_all(df, tree_dir, navigation_path, out_path, model):
    global _current_doc_name, _current_financebench_id, _current_stage
    out_path.parent.mkdir(parents=True, exist_ok=True)
    nav_records = {r["financebench_id"]: r for r in _load_jsonl(navigation_path)}
    completed = _load_completed_ids(out_path)
    tree_cache = {}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            continue
        if fb_id not in nav_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- no navigation output yet")
            continue

        print(f"[{i}/{len(df)}] {fb_id}: generating ...")
        _current_doc_name, _current_financebench_id, _current_stage = row.doc_name, fb_id, "generation"
        try:
            if row.doc_name not in tree_cache:
                tree_cache[row.doc_name] = load_tree(tree_dir, row.doc_name)
            tree = tree_cache[row.doc_name]
            node_map = create_node_mapping(tree["structure"])
            sections_text = fetch_node_text(nav_records[fb_id]["node_ids"], node_map, tree["page_texts"])
            answer = generate_answer(row.question, sections_text, model)
            record = {"financebench_id": fb_id, "doc_name": row.doc_name, "model_answer": answer}
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
        except Exception as e:
            print(f"    FAILED: {e}")


run_generation_all(df, TREE_DIR, NAVIGATION_PATH, GENERATION_PATH, MODEL)

---
## Stage 5 — Preview

No scoring yet (see the scope note in Stage 0) — just a look at what's been generated so far, next to the gold answer.

In [ ]:
generation_records = pd.DataFrame(_load_jsonl(GENERATION_PATH))
preview = df.merge(generation_records, on=["financebench_id", "doc_name"])[
    ["financebench_id", "doc_name", "question", "answer", "model_answer"]
]
print(f"{len(preview)}/{len(df)} questions have a generated answer so far")
preview.head(10)

---
## Stage 6 — Score each answer

Deterministic numeric matching first (handles unit differences and small
rounding); falls through to an LLM judge (`openai/gpt-oss-120b` via Groq's
free tier — a different model family from Gemini, which matters for avoiding
self-enhancement bias) only when a question isn't numeric. Reuses
`evaluation/answer_scorer.py`, the same scorer `vector_rag_pipeline.ipynb`
uses, so both pipelines are graded identically. Spot-check 10-15% of judge
outputs by hand before trusting them for the real experiment.

**Resumable** — needs Stage 4 to have produced a generated answer for the
question first.

In [ ]:
from groq import Groq
from evaluation.answer_scorer import score_answer

judge_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
JUDGE_MODEL = "openai/gpt-oss-120b"


# For each question with a generated answer but no score yet: run the
# deterministic matcher, fall back to the LLM judge if that's inconclusive,
# save the label. Judge token usage AND latency (when the judge was actually
# called -- the deterministic path makes no API call, so timing it would just
# log ~0s noise) gets logged to cost_tracker same as everything else.
def run_scoring_all(df, generation_path, out_path, judge_client, judge_model):
    global _current_doc_name, _current_financebench_id, _current_stage
    out_path.parent.mkdir(parents=True, exist_ok=True)
    gen_records = {r["financebench_id"]: r for r in _load_jsonl(generation_path)}
    completed = _load_completed_ids(out_path)

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            continue
        if fb_id not in gen_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- no generated answer yet")
            continue

        print(f"[{i}/{len(df)}] {fb_id}: scoring ...")
        _current_doc_name, _current_financebench_id, _current_stage = row.doc_name, fb_id, "judge"
        try:
            model_answer = gen_records[fb_id]["model_answer"]
            t0 = time.time()
            result, usage = score_answer(
                row.question, row.answer, model_answer,
                judge_client=judge_client, judge_model=judge_model,
            )
            judge_latency = time.time() - t0
            if usage:
                cost_tracker.log(
                    pipeline="vectorless_rag", stage="judge", model=judge_model,
                    input_tokens=usage["input_tokens"], output_tokens=usage["output_tokens"],
                    doc_name=row.doc_name, financebench_id=fb_id,
                    latency_sec=judge_latency,
                )
            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name,
                "label": result.label, "method": result.method, "reasoning": result.reasoning,
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
        except Exception as e:
            print(f"    FAILED: {e}")


run_scoring_all(df, GENERATION_PATH, SCORING_PATH, judge_client, JUDGE_MODEL)


---
## Stage 7 — Retrieval quality (Recall@k, MRR@k)

Only meaningful for vector RAG and vectorless RAG (long-context has no
retrieval step) — see CLAUDE.md. Converts Stage 3's ranked `node_ids` into a
ranked page list (a section can span several pages), then compares against
FinanceBench's gold `evidence_page_num` using `evaluation/retrieval_metrics.py`
— the same metric code `vector_rag_pipeline.ipynb` uses, so both pipelines'
retrieval quality is directly comparable.

No LLM calls here, no cost — purely a comparison against already-saved
navigation output.

**Resumable** — needs Stage 3's navigation output for the question.

In [ ]:
from dataclasses import asdict
from evaluation.retrieval_metrics import compute_retrieval_metrics


# A node can span several physical pages -- flatten a ranked node_id list into
# a ranked page list (first occurrence wins), 0-indexed to match
# evidence_page_num. start_index/end_index are 1-indexed and inclusive.
def ranked_node_ids_to_ranked_pages(node_ids, node_map) -> list:
    ranked_pages = []
    for node_id in node_ids:
        node = node_map.get(node_id)
        if node is None:
            continue
        start, end = node["start_index"] - 1, node["end_index"] - 1
        for page in range(start, end + 1):
            if page not in ranked_pages:
                ranked_pages.append(page)
    return ranked_pages

In [ ]:
# For each question with navigation output but no retrieval metrics yet:
# convert its node_ids to a ranked page list, compare against gold
# evidence_page_num, save.
def run_retrieval_metrics_all(df, tree_dir, navigation_path, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    nav_records = {r["financebench_id"]: r for r in _load_jsonl(navigation_path)}
    completed = _load_completed_ids(out_path)
    tree_cache = {}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            continue
        if fb_id not in nav_records:
            continue

        print(f"[{i}/{len(df)}] {fb_id}: computing retrieval metrics ...")
        try:
            if row.doc_name not in tree_cache:
                tree_cache[row.doc_name] = load_tree(tree_dir, row.doc_name)
            node_map = create_node_mapping(tree_cache[row.doc_name]["structure"])
            ranked_pages = ranked_node_ids_to_ranked_pages(nav_records[fb_id]["node_ids"], node_map)
            gold_pages = [e["evidence_page_num"] for e in row.evidence]
            metrics = compute_retrieval_metrics(ranked_pages, gold_pages, fb_id, row.doc_name, stage="navigation")
            with out_path.open("a") as f:
                f.write(json.dumps(asdict(metrics)) + "\n")
        except Exception as e:
            print(f"    FAILED: {e}")


run_retrieval_metrics_all(df, TREE_DIR, NAVIGATION_PATH, RETRIEVAL_PATH)

---
## Stage 8 — Dedicated latency timing pass

A small, **strictly sequential** pass over a fixed sample, run fully fresh
each time (navigate → fetch section text → generate, timed end-to-end and
per sub-stage) — matching the project brief's "run each query multiple
times, report the median." Tree-building (Stage 2) is excluded: that's
one-time preprocessing per document, not something a live query pays for, so
a question can only be sampled here if its document is already indexed.
Scoring is excluded too — grading isn't part of response time.

**Costs real API calls beyond Stages 2-4** — every (question, repeat) pair
here is a fresh navigate+generate call, not reused from earlier stages.
Default is a conservative **5 questions × 3 repeats = 15 timed passes** —
raise `LATENCY_SAMPLE_SIZE`/`LATENCY_REPEATS` only once you're ready to spend
more on this specific stage. It's resumable in the sense that a
(question, repeat) pair already timed is skipped, so re-running this cell
without changing the sample doesn't cost anything more.

In [ ]:
import time
import statistics

LATENCY_SAMPLE_SIZE = 5   # questions -- kept small deliberately, raise when ready to spend more
LATENCY_REPEATS = 3       # per project brief: run each query multiple times, report the median


# One fully-fresh, uncached pass: navigate -> fetch selected section text ->
# generate. Returns (timings_sec dict, model_answer).
def time_single_pass(row, tree, node_map, model):
    global _current_doc_name, _current_financebench_id, _current_stage
    _current_doc_name, _current_financebench_id = row.doc_name, row.financebench_id
    timings = {}
    t_total0 = time.time()

    _current_stage = "latency_navigation"
    t0 = time.time()
    node_ids = navigate_tree(row.question, tree["structure"], model)
    timings["navigate"] = time.time() - t0

    t0 = time.time()
    sections_text = fetch_node_text(node_ids, node_map, tree["page_texts"])
    timings["fetch_section_text"] = time.time() - t0

    _current_stage = "latency_generation"
    t0 = time.time()
    answer = generate_answer(row.question, sections_text, model)
    timings["generate"] = time.time() - t0

    timings["total"] = time.time() - t_total0
    return timings, answer

In [ ]:
# Samples a fixed set of already-indexed questions and times each one
# `repeats` times, skipping (question, repeat) pairs already timed.
def run_latency_pass(df, tree_dir, model, out_path, sample_size=LATENCY_SAMPLE_SIZE, repeats=LATENCY_REPEATS, seed=42):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    indexed = df[df.doc_name.apply(lambda d: is_tree_built(tree_dir, d))]
    if len(indexed) < len(df):
        print(f"Note: {len(df) - len(indexed)} questions excluded -- their document isn't indexed yet")
    sample = indexed.sample(n=min(sample_size, len(indexed)), random_state=seed)

    completed_pairs = {(r["financebench_id"], r["repeat"]) for r in _load_jsonl(out_path)}
    total_runs = len(sample) * repeats
    print(f"Timing {len(sample)} questions x {repeats} repeats = {total_runs} passes "
          f"({len(completed_pairs)} already done)")

    tree_cache = {}
    for row in sample.itertuples():
        if row.doc_name not in tree_cache:
            tree = load_tree(tree_dir, row.doc_name)
            tree_cache[row.doc_name] = (tree, create_node_mapping(tree["structure"]))
        tree, node_map = tree_cache[row.doc_name]

        for repeat in range(1, repeats + 1):
            if (row.financebench_id, repeat) in completed_pairs:
                continue
            print(f"  {row.financebench_id} repeat {repeat}/{repeats} ...")
            try:
                timings, answer = time_single_pass(row, tree, node_map, model)
                record = {
                    "financebench_id": row.financebench_id, "doc_name": row.doc_name,
                    "repeat": repeat, "timings_sec": timings, "model_answer": answer,
                }
                with out_path.open("a") as f:
                    f.write(json.dumps(record) + "\n")
            except Exception as e:
                print(f"    FAILED: {e}")


run_latency_pass(df, TREE_DIR, MODEL, LATENCY_PATH)

In [ ]:
# Median-of-N per question (per the project brief), then median across
# questions for the headline number -- robust to one slow/retried pass
# dragging the number the way a mean would.
def summarize_latency(out_path):
    records = _load_jsonl(out_path)
    if not records:
        print(f"No latency data yet in {out_path}")
        return {}

    by_question = {}
    for r in records:
        by_question.setdefault(r["financebench_id"], []).append(r["timings_sec"]["total"])
    per_question_median = {fb_id: statistics.median(vals) for fb_id, vals in by_question.items()}
    overall_median = statistics.median(per_question_median.values())

    stage_names = [k for k in records[0]["timings_sec"] if k != "total"]
    stage_medians = {
        stage: statistics.median(r["timings_sec"][stage] for r in records) for stage in stage_names
    }
    print(f"Median end-to-end latency ({len(per_question_median)} questions): {overall_median:.2f}s")
    for stage, med in stage_medians.items():
        print(f"  median {stage}: {med:.2f}s")
    return {"overall_median_sec": overall_median, "stage_medians_sec": stage_medians}


summarize_latency(LATENCY_PATH)

---
## Stage 9 — Summarize

Answer-quality breakdown (Stage 6), retrieval Recall@k/MRR@k (Stage 7), and
token/cost totals by stage (Stage 2's indexing calls onward) — computed over
whatever's done so far in each stage's output file, not requiring all 150
questions to be finished.

In [ ]:
from collections import Counter
from evaluation.retrieval_metrics import aggregate_retrieval_metrics, RetrievalMetrics, K_VALUES


def summarize_results(scoring_path, retrieval_path, cost_path):
    scoring = _load_jsonl(scoring_path)
    if scoring:
        n = len(scoring)
        label_counts = Counter(r["label"] for r in scoring)
        print(f"Answer quality ({n} scored questions):")
        for label, count in label_counts.items():
            print(f"  {label}: {count} ({100 * count / n:.1f}%)")
    else:
        print(f"No scored results yet in {scoring_path}")

    print()
    retrieval = _load_jsonl(retrieval_path)
    if retrieval:
        # JSON object keys are always strings -- recall_at_k/mrr_at_k need
        # their keys back as ints (5, 10, 15) to match K_VALUES.
        records = [
            RetrievalMetrics(
                r["financebench_id"], r["doc_name"], r["stage"],
                {int(k): v for k, v in r["recall_at_k"].items()},
                {int(k): v for k, v in r["mrr_at_k"].items()},
            )
            for r in retrieval
        ]
        agg = aggregate_retrieval_metrics(records)
        print(f"Retrieval quality ({agg['n_questions']} questions):")
        for k in K_VALUES:
            print(f"  Recall@{k}: {agg['recall_at_k'][k]:.3f}   MRR@{k}: {agg['mrr_at_k'][k]:.3f}")
    else:
        print(f"No retrieval metrics yet in {retrieval_path}")

    print()
    costs = _load_jsonl(cost_path)
    if costs:
        tokens_by_stage = Counter()
        cost_by_stage = Counter()
        for r in costs:
            tokens_by_stage[r["stage"]] += r["input_tokens"] + r["output_tokens"]
            if r["cost_usd"] is not None:
                cost_by_stage[r["stage"]] += r["cost_usd"]
        print(f"Tokens/cost by stage ({len(costs)} logged calls):")
        for stage, tokens in tokens_by_stage.items():
            cost_str = f"${cost_by_stage[stage]:.4f}" if stage in cost_by_stage else "no price set"
            print(f"  {stage}: {tokens:,} tokens, {cost_str}")
        print(f"  TOTAL: ${sum(cost_by_stage.values()):.4f}")
    else:
        print(f"No cost data yet in {cost_path}")


summarize_results(SCORING_PATH, RETRIEVAL_PATH, COSTS_PATH)